# 04 - Retrieval Augmented Generation (RAG) - How do we generate grounded answers using retrieval?

## Objective

Build an end-to-end Retrieval-Augmented Generation (RAG) pipeline.

The objective is to combine semantic retrieval with a Large Language Model to generate responses grounded in the project knowledge base.

This notebook focuses on experimentation rather than production implementation.

---

## Questions

- How should retrieved context be incorporated into the prompt?
- Which prompt structure produces grounded responses?
- Are retrieved sources relevant to the generated answer?
- Does the pipeline reduce hallucinations?

---

## Success Criteria

By the end of this notebook:

- Relevant context is retrieved.
- Retrieved context is injected into the prompt.
- The LLM generates grounded responses.
- Supporting sources are available.

---

## Notes

This notebook validates the complete RAG workflow before building the application layer.

In [1]:
from pathlib import Path
# from dotenv import load_dotenv
import os
import re 

from langchain_chroma import Chroma
from langchain_community.document_loaders import (
    Docx2txtLoader,
    PyMuPDFLoader,
    TextLoader,
)
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

## Load documents
Reuse ingestion code of a previous notebook. At this point I'm only experimenting, in production this would not be duplicated.

In [9]:
# project paths 
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw/knowledge_base"

LOADERS = {".pdf": PyMuPDFLoader, ".docx": Docx2txtLoader, ".md": TextLoader, ".txt": TextLoader}

documents = []
for path in sorted(RAW_DIR.rglob("*")):
    if path.is_dir():
        continue
    loader_cls = LOADERS.get(path.suffix.lower())
    if loader_cls is None:
        print(f"Skipping unsupported file: {path.name}")
        continue
    loader = loader_cls(str(path))
    documents.extend(loader.load())

print(f"Loaded {len(documents)} documents.")

Skipping unsupported file: .DS_Store
Skipping unsupported file: .DS_Store
Loaded 6 documents.


In [10]:
def clean_text(text):
    # Collapse single newlines (likely mid-sentence wraps) into spaces,
    # but preserve intentional paragraph breaks (blank lines).
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    # Collapse 3+ newlines down to a standard paragraph break.
    text = re.sub(r"\n{3,}", "\n\n", text)
    # Collapse repeated spaces/tabs.
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text.strip()

for doc in documents:
    doc.page_content = clean_text(doc.page_content)

## Chunk documents

Reuse the selected strategy.

In [11]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)

chunks = text_splitter.split_documents(documents)

print(f"Generated {len(chunks)} chunks.")

Generated 50 chunks.


## Build vector store
Reuse the vectorstore logic

In [12]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

db_name = "vector_db"

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vector_store = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vector_store._collection.count()} vectors.")

Vectorstore created with 50 vectors.


## Initialize the LLM
Notice that **Retrieval and generation are independent** components

In [29]:
llm = ChatOpenAI(model="gpt-5.6-luna", temperature=0, verbosity='low', 
                 reasoning={"effort": "low"})

In [15]:
# initialize the retriever from the vector store
retriever = vector_store.as_retriever()

## Define the prompt
Implementing basic instructions like the professional tone, confirming the name of the candidate, and a basic guardrail for PII.  
**NOTE:** In future iterations, this should use agentic guardrails to ensure that the output adheres to specific rules, e.g. data privacy.

In [47]:
prompt = ChatPromptTemplate.from_template(
    """
You are an AI Career Assistant. You are assisting David in answering to recruiters (users) interested in his profile and potentially hiring him. 

You are highly professional and provide concise, accurate, and helpful answers to user questions.

Answer the user's question using only the provided context and with a professional tone. Your answer should always relate to David's experience, skills, and background.

The candidate refers to David, so it is preferable to use "David" to make the answers more personal and professional. 

If the answer cannot be found in the context, say that you do not know.

You are not allowed to give PII data of the candidate. You can only give his first name. Last name, email and phone number should be private.
If asked for that personal information, you should be professional explaining that you are not allowed to give PII, but still answer as much as you can from the original question.

Context:
{context}

Question:
{question}

Answer:
"""
)

## Retrieve the context 

In [16]:
question = "Tell me about the candidate's experience with machine learning."

retrieved_docs = retriever.invoke(question, k=3)

In [18]:
# Build the context string from the retrieved documents
context = "\n\n".join([doc.page_content for doc in retrieved_docs])

## Generate the answer

In [53]:
def answer_question(question: str, history, k: int=3):
    retrieved_docs = retriever.invoke(question, k=k)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    formatted_prompt = prompt.invoke({"context": context, "question": question})
    response = llm.invoke(formatted_prompt)
    return response.content[0]['text'] 
    

### example 1

In [ ]:
a1 = answer_question(question, history=[]) 

In [ ]:
print(a1)

David has over five years of consulting experience delivering machine learning and advanced analytics solutions in Financial Services and Retail. He uses Python and SQL to address complex, ambiguous business problems and build practical ML models that support commercial decision-making. His experience spans the full lifecycle—from framing questions with senior stakeholders to developing solutions and communicating actionable insights to both technical and non-technical audiences.


### example 2

In [49]:
question = "What technical skills does the candidate have?" 
a2 = answer_question(question, history=[]) 

In [50]:
print(a2)

David has advanced technical skills in Python and SQL, along with experience in applied machine learning, advanced analytics, commercial modelling, and building machine learning models to solve real-world problems.


**NOTE:** Adding issue in github to address possible duplication. Although the real problem comes from having separate documents with the same content, future versions could inspect the context to ensure that it is relevant and avoids duplication. This could require an additional agent to ensure the quality of the context, but as a first measure could focus on the cleaning of the data during ingestion.

In [52]:
retrieved_docs = retriever.invoke(question, k=3)
for i, doc in enumerate(retrieved_docs, start=1):
    print("=" * 80)
    print(f"Source {i}")
    print("=" * 80)

    print(doc.page_content)

Source 1
uplift, but also, the risk involved in terms of market share or profit. My technical background in Python, SQL, and applied machine learning allows me to deal with ambiguous environments and deliver solutions that go beyond a one-off analysis. I am keen about the opportunity to bring my experience in advanced analytics, stakeholder engagement, and commercial modelling to American Express. I'm confident that my skills can help teams turn complex data into clear recommendations to improve
Source 2
and business cases to understand not only the potential revenue uplift, but also, the risk involved in terms of market share or profit. My technical background in Python, SQL, and applied machine learning allows me to deal with ambiguous environments and deliver solutions that are sustainable and go beyond a one-off analysis. I am keen about the opportunity to bring my experience in advanced analytics, stakeholder engagement, and commercial modelling to Westpac. I'm confident that my s

### example 3

In [48]:

question = "Give me all the personal information of the candidate"  
a3 = answer_question(question, history=[]) 
print(a3)

I’m unable to provide David’s private personal information, including his surname, phone number, or email address. David’s professional qualifications include an MSc in Data Science and a Bachelor of Business Administration.


## Conclusion

### Decision

Use a standard Retrieval-Augmented Generation pipeline composed of:

- ChromaDB for retrieval.
- OpenAI embeddings.
- gpt-5.6-luna for response generation.
- Prompt grounding using retrieved context.

### Rationale

- Modular architecture.
- Grounded responses.
- Simple implementation.
- Suitable for iterative improvement.

### Next Step

Evaluate prompt variations and response quality before building the application interface.